# Precprocess the data

## Point source masking

The CIB map is to have sources exceeding $2\;\text{mJy}$ masked with single pixel masks.

## Cluster masking

The tSZ map is to have clusters with mass $M_{500}\ge3\times10^{14}$ are to be masked with radii ranging from $3\theta_{500}$ to $5\theta_{500}$ with a minimum radius of 10 arcminutes.

## Masking with sigma clipping

Until we get access to cluster and point source masks, we use sigma clipping. The process is to compute $\mu$ and $\sigma$, remove all pixels above $\mu+10\times\sigma$ and iterate until no more pixels are removed by this process. Input maps have an NSIDE of 8192, corresponding to 805306368 pixels that are 0.43 arcminutes.

In [ ]:
import numpy as np
import astropy.units as u
import healpy as hp
import os
import matplotlib.pyplot as plt

In [ ]:
data_dir = 'data'
plots_dir = 'plots'
os.makedirs(plots_dir, exist_ok=True)

cib_map = hp.read_map(os.path.join(data_dir, 'mdpl2_len_mag_cibmap_act_150_uk.fits'), memmap=True)

hp.visufunc.mollview(cib_map, title='CIB Map at 150 GHz', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_150ghz.png'), dpi=300)

tsz_map = hp.read_map(os.path.join(data_dir, 'mdpl2_ltszNG_bahamas80_rot_sum_4_176_bnd_unb_1.0e+12_1.0e+18_v103021_lmax24000_nside8192_interp1.0_method1_1_lensed_map.fits'), memmap=True)

hp.visufunc.mollview(tsz_map, title='tSZ Map at 150 GHz', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_150ghz.png'), dpi=300)

# Get pixel info:
nside = hp.get_nside(cib_map)
npix = hp.nside2npix(nside)
pix_size = hp.nside2resol(nside, arcmin=True)
print(f"NSIDE: {nside}, NPIX: {npix}, Pixel Size: {pix_size:.2f} arcmin")

# Plot histogram of pixel values:
harsh_clip = 1e-5
cib_map_clipped = np.clip(cib_map, -harsh_clip, harsh_clip)
plt.figure(figsize=(10, 6))
plt.hist(cib_map_clipped.flatten(), bins=100, alpha=0.5, label='CIB Map')
plt.hist(tsz_map.flatten(), bins=100, alpha=0.5, label='tSZ Map')
plt.xlabel('Temperature (K_CMB)')
plt.ylabel('Number of Pixels')
plt.yscale('log')
plt.title('Pixel Value Distribution')
plt.legend()
plt.savefig(os.path.join(plots_dir, 'pixel_value_histogram.png'), dpi=300)

In [ ]:
# Get CIB mask using sigma clipping
from astropy.stats import sigma_clip
cib_mask = sigma_clip(cib_map, sigma=10, maxiters=5).mask
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_ptsrc_mask.fits'), cib_mask.astype(float), overwrite=True)
hp.visufunc.mollview(cib_mask.astype(float), title='CIB Mask (Sigma Clipped)', unit='Mask')
plt.savefig(os.path.join(plots_dir, 'cib_ptsrc_mask.png'), dpi=300)

cib_150_masked = np.where(cib_mask, 0, cib_map)
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_150_ptsrc_masked.fits'), cib_150_masked, overwrite=True)
hp.visufunc.mollview(cib_150_masked, title='CIB Map with Mask Applied', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_ptsrc_masked.png'), dpi=300)

# Get tSZ mask using sigma clipping
tsz_mask = sigma_clip(tsz_map, sigma=10, maxiters=5).mask
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_ptsrc_mask.fits'), tsz_mask.astype(float), overwrite=True)
hp.visufunc.mollview(tsz_mask.astype(float), title='tSZ Mask (Sigma Clipped)', unit='Mask')
plt.savefig(os.path.join(plots_dir, 'tsz_ptsrc_mask.png'), dpi=300)

tsz_150_masked = np.where(tsz_mask, 0, tsz_map)
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked.fits'), tsz_150_masked, overwrite=True)
hp.visufunc.mollview(tsz_150_masked, title='tSZ Map with Mask Applied', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_ptsrc_masked.png'), dpi=300)


## Low pass filter

Both maps have a low pass filter applied, with $\ell\ge7000$ set to zero.

In [ ]:
# Low pass both maps with ell > 7000 set to zero
lmax = 7000
cib_150_lowpassed = hp.sphtfunc.alm2map(hp.sphtfunc.map2alm(cib_150_masked, lmax=lmax), nside=nside)
tsz_150_lowpassed = hp.sphtfunc.alm2map(hp.sphtfunc.map2alm(tsz_150_masked, lmax=lmax), nside=nside)
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_150_ptsrc_masked_lowpassed.fits'), cib_150_lowpassed, overwrite=True)
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked_lowpassed.fits'), tsz_150_lowpassed, overwrite=True)
hp.visufunc.mollview(cib_150_lowpassed, title='CIB Map Lowpassed (l > 7000)', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_lowpassed.png'), dpi=300)
hp.visufunc.mollview(tsz_150_lowpassed, title='tSZ Map Lowpassed (l > 7000)', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_lowpassed.png'), dpi=300)

## Creating map patches

In [ ]:
import astropy.units as u

RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg

@u.quantity_input
def get_patch_centers(gal_cut: u.deg, step_size: u.deg):
    """ Function to get the centers of the various patches to be cut out.

    Parameters
    ----------
    gal_cut: float
        We will miss out the region +/- `gal_cut` in Galactic latitude, measured
        in degrees.
    step_size: float
        Stepping distance in Galactic longitude, measured in degrees, between 
        patches.

    Returns
    -------
    list(tuple(float))
        List of two-element tuples containing the longitude and latitude.
    """
    gal_cut = gal_cut.to(u.deg)
    step_size = step_size.to(u.deg)
    assert gal_cut.unit == u.deg
    assert step_size.unit == u.deg
    southern_lat_range = np.arange(-90, (-gal_cut-step_size).value, step_size.value) * u.deg
    northern_lat_range = np.arange((gal_cut + step_size).value, 90, step_size.value) * u.deg
    lat_range = np.concatenate((southern_lat_range, northern_lat_range))

    centers = []
    for t in lat_range:
        step = step_size.value / np.cos(t.to(u.rad).value)
        for i in np.arange(0, 360, step):
            centers.append((i * u.deg, t))
    return centers

class FlatCutter(object):
    """ Object to control the extraction of flat patches from a given HEALPix 
    map.

    Object is initialized with parameters defining the geometry of the patch:
    its length in degrees, and the number of pixels in each direction. 
    The `rotate_and_interpolate` method defines a grid centered at (0, 0)
    of dimensions corresponding to `xlen`, `ylen`, and rotates it to the 
    point (lon, lat). The value of the map at the resulting grid of longitudes 
    and latitudes is then determined by interpolation. 
    """
    @u.quantity_input
    def __init__(self, ang_x: u.deg, ang_y: u.deg, xres, yres):
        assert type(xres) is int
        assert type(yres) is int
        self.xres = xres
        self.yres = yres

        self.ang_x = ang_x
        self.ang_y = ang_y
        
        # get grid of unit vectors corresponding to flat patch around
        # pole (z = 1). For this we use ang_x in radians, as is appropriate
        # for the implicit small-angle approximation 
        self.xarr = np.linspace(- self.ang_x.to(u.rad).value / 2., 
                                self.ang_x.to(u.rad).value / 2., xres)
        self.yarr = np.linspace(- self.ang_y.to(u.rad).value / 2., 
                                self.ang_y.to(u.rad).value / 2., yres)

        xgrid, ygrid = np.meshgrid(self.xarr, self.yarr)
        xgrid = xgrid.ravel()[None, :]
        ygrid = ygrid.ravel()[None, :]
        zgrid = np.ones_like(ygrid)
        
        # vectors corresponding to cartesian grid around poll
        self.vecs = np.concatenate((xgrid, ygrid, zgrid)).T

        # get the latitude (*not colatitude*) and longitude in degrees
        # of the cartesian grid points around the pole. 
        self.lons, self.lats = hp.vec2ang(self.vecs, lonlat=True)
        self.lats *= u.deg
        self.lons *= u.deg
        return
    
    @u.quantity_input
    def rotate_to_pole_and_interpolate(self, lon: u.deg, lat: u.deg, ma):
        """ Method to rotate the grid at (0, 0) to `rot=(lon, lat)`, and sample
        the map at the grid points by interpolation.

        Parameters
        ----------
        lat, lon: float
            Latitude (*not* colatitude) and longitude of point to be rotated
            to the North pole, in degrees.
        ma: ndarray
            Healpix map from which the interpolation is to be made.
        """
        if hp.pixelfunc.maptype(ma) == 0:  # a single map is converted to a list
            ma = [ma]
        # define a rotation object in terms of the theta_rot and phi_rot angles.
        # This returns a rotator object that can be applied to rotate a given
        # vector by this angle. Since we are interested in rotating some patch
        # to the pole, we actually want to apply the *inverse* rotation operator
        # to the vectors self.co_lats, self.lons.
        lon = lon.to(u.deg)
        lat = lat.to(u.deg)
        rotator = hp.Rotator(rot=[lon.value, lat.value - 90.], deg=True)
        self.inv_lon_grid, self.inv_lat_grid = rotator.I(self.lons.to(u.deg).value, self.lats.to(u.deg).value, lonlat=True)
        # Interpolate the original map to the pixels centers in the new ref frame
        m_rot = [hp.get_interp_val(each, self.inv_lon_grid, self.inv_lat_grid, lonlat=True) for each in ma]

        # Rotate polarization
        if len(m_rot) > 1:
            # Create a complex map from QU  and apply the rotation in psi due to the rotation
            # Slice from the end of the array so that it works both for QU and IQU
            m_rot[-2], m_rot[-1] = spin2rot(m_rot[-2], m_rot[-1], rotator.angle_ref(self.inv_lon_grid, self.inv_lat_grid, lonlat=True))
            m_rot[-2], m_rot[-1] = spin2rot(m_rot[-2], m_rot[-1], self.lons.to(u.rad).value)
        else:
            m_rot = m_rot[0]
        return np.moveaxis(np.array(m_rot).reshape(-1, self.xres, self.yres), 0, -1)

In [ ]:
RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg
ptsrc = 2
flatskymapparams = [RES, RES, 60.*ANG_X.value/RES, 60.*ANG_Y.value/RES] #Code requires pixelres to be in arcmin
flatskymapparams

In [ ]:
final_cib_150_map = hp.fitsfunc.read_map(os.path.join(data_dir, 'cib_150_ptsrc_masked_lowpassed.fits'))
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cib_cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, final_cib_150_map) for (lon, lat) in centers], dtype=np.float32)

fname = f"data/low_pass/{ptsrc}mJy/cut_maps_RES_{RES}_ANG_X_{ANG_X.value}_deg_{ptsrc}mJy_lp.npy"
np.save(fname,cib_cut_maps)

def apply_maxmin_normalization(maps):
    min_val = np.min(maps)
    max_val = np.max(maps)
    return (maps - min_val) / (max_val - min_val) 

processed_maps = np.copy(cut_maps)

min_val = np.min(cut_maps)
max_val = np.max(cut_maps)
print (min_val,max_val - min_val)

processed_maps = apply_maxmin_normalization(processed_maps)

final_tsz_map = hp.fitsfunc.read_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked_lowpassed.fits'))

In [ ]:
processed_maps = apply_maxmin_normalization(processed_maps)
fpath = "data/low_pass/{:d}mJy/CIB_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_zero_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
#fpath = "data/low_pass/{:d}mJy/tSZ3_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_norm_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
print(fpath)
np.save(fpath, processed_maps)

## Gaussian patches

Create patches from the same power spectrum as the AGORA CIB map.

In [ ]:
cib_map_meansub = final_cib_150_map - np.mean(cib_150_map)
cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)
gaussian_map = hp.synfast(cl_from_map, 8192)
centers = get_patch_centers(gal_cut=0*u.deg, step_size=STEP_SIZE)
print("Total patches: ", len(centers))
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolator(lon, lat, gaussian_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz.npy", cut_maps)

## Joint Gaussian maps

Record auto and cross power spectra and patches from original AGORA CIB and tSZ maps.

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cib_cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)
tsz_map_meansub = tsz_150_map - np.mean(tsz_150_map)
tsz_cl_from_map = hp.anafast(tsz_map_meansub, lmax=13000)
cib_tsz_cl_from_map = hp.anafast(cib_map_meansub,tsz_map_meansub, lmax=13000)
hp.write_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits",tsz_cl_from_map)
hp.write_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits",cib_cl_from_map)
hp.write_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits",cib_tsz_cl_from_map)
cib_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits")
tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits")
cib_tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits")
# Ordering for healpy.synfast:
# [C_ℓ^TT (aa), C_ℓ^EE (bb), C_ℓ^BB (ignored), C_ℓ^TE (ab)]
cls = [cib_cl_from_map, tsz_cl_from_map, np.zeros_like(cib_cl_from_map), cib_tsz_cl_from_map]
maps = hp.synfast(cls, nside=8192, new=True, pol=False)
cib_map = maps[0]
tsz_map = maps[1]
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, cib_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_cib_joint3.npy",cut_maps)
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, tsz_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz_joint3.npy",cut_maps)